# 09. Signal Propagation (Heat Diffusion)
This notebook constructs the sparse Graph Laplacian ($\mathcal{L}_E = \mathbf{D} - \mathbf{W}_E$) and simulates the spatial heat diffusion of the exerkine signal ($\mathbf{F}(t) = e^{-t\mathcal{L}_E}\mathbf{f}_0$) as specified in Sections 11, 12, and 13 of the [EXERKINEMAP Mathematical Model](https://github.com/gomezdj/exerkinemap/blob/main/mathematical_model.md).

In [ ]:
import numpy as np
import pandas as pd
import scanpy as sc
from scipy.sparse import coo_matrix, diags
from scipy.sparse.linalg import expm_multiply
from pathlib import Path

PROJECT_ROOT = Path('.').resolve().parents[0]
PROCESSED_SPATIAL_DIR = PROJECT_ROOT / 'data' / 'processed' / 'spatial'
NETWORK_DIR = PROJECT_ROOT / 'data' / 'processed' / 'networks'

print(f'Project Root Directory: {PROJECT_ROOT}')

## Constructing Graph Laplacian ($\mathcal{L}_E$) & Simulating Diffusion ($\mathbf{F}(t)$)

In [ ]:
spatial_path = PROCESSED_SPATIAL_DIR / 'motrpac_spatial_propagated.h5ad'
network_path = NETWORK_DIR / 'spatial_communication_network.csv'

if spatial_path.exists() and network_path.exists():
    adata = sc.read_h5ad(spatial_path)
    spatial_network = pd.read_csv(network_path)
    
    # Aggregate edge weights W_E
    edge_weights = spatial_network.groupby(['sender_spot', 'receiver_spot'])['S_tilde_score'].sum().reset_index()
    spot_to_idx = {name: idx for idx, name in enumerate(adata.obs_names)}
    
    row_idx = edge_weights['sender_spot'].map(spot_to_idx).dropna().values.astype(int)
    col_idx = edge_weights['receiver_spot'].map(spot_to_idx).dropna().values.astype(int)
    weights = edge_weights['S_tilde_score'].values[:len(row_idx)]
    
    num_spots = adata.n_obs
    W_E = coo_matrix((weights, (row_idx, col_idx)), shape=(num_spots, num_spots)).tocsr()
    
    # Laplacian L_E = D - W_E
    out_degrees = np.array(W_E.sum(axis=1)).flatten()
    D = diags(out_degrees, format='csr')
    L_E = D - W_E
    
    print(f'Graph Laplacian constructed. Shape: {L_E.shape}, Non-zeros: {L_E.nnz}')
else:
    print('Required spatial network or propagated h5ad dataset not found. Ensure workflow scripts 09 and 10 are executed.')